# B6 - Final test-set evaluation

Lock the test split (2025 Q1) and compare the calibrated CatBoost
(from B5) against the baselines (B2) on the held-out window. This is
the only place we touch the test split - anywhere else and we'd be
overfitting to it.

We report:

* Pinball loss, MAE, coverage by horizon group.
* Segment breakdowns (peak/off-peak, weekday/weekend, crisis/post-crisis,
  negative-price events, top-decile spikes).
* Diebold-Mariano significance vs the seasonal-naive-168h baseline.
* A single in-notebook leaderboard.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

## Load all three splits

In [ ]:
from data.loaders import load_split
from module_b.features import prepare_supervised, HORIZON_COL
from module_b.features import build_features

train_df = load_split('train')
val_df = load_split('val')
test_df = load_split('test')
full = pd.concat([train_df, val_df, test_df])

feat = build_features(full, bundles=('calendar', 'lags', 'fundamentals',
                                     'spike', 'regime', 'weather'))
past_cols = [c for c in feat.columns if
             c.startswith('price_lag') or c.startswith('price_rmean')
             or c.startswith('price_rstd')
             or c in ('residual_load', 'renewable_penetration',
                      'clean_spark_anchor', 'clean_dark_anchor',
                      'is_high_residual_load', 'is_renewable_scarcity',
                      'crisis_2022_flag')]
future_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
               'is_weekend', 'is_holiday_DE',
               'weather_mean_wind_speed_100m', 'weather_mean_shortwave_radiation']
X, y = prepare_supervised(feat, horizons=range(1, 25),
                          past_cols=[c for c in past_cols if c in feat.columns],
                          future_cols=[c for c in future_cols if c in feat.columns])

is_train_val = X['origin_ts'] < test_df.index.min()
is_test = ~is_train_val
X_trainval, y_trainval = X.loc[is_train_val], y.loc[is_train_val]
X_test_full, y_test_full = X.loc[is_test], y.loc[is_test]
print(f'train+val={len(X_trainval):,}, test={len(X_test_full):,}')

## Fit baselines + calibrated CatBoost

In [ ]:
from module_b.models import (
    NaiveForecaster,
    SeasonalNaiveForecaster,
    CatBoostQuantileForecaster,
    LightGBMQuantileForecaster,
)
from module_b.evaluation import ConformalQuantileRegressor

price_series = full['price']

naive = NaiveForecaster().fit(X_trainval, y_trainval)
sn168 = SeasonalNaiveForecaster(season_hours=168).fit(
    X_trainval, y_trainval, price_series=price_series)

# Fit on pre-2024, calibrate on 2024 (val), evaluate on 2025 Q1 (test).
is_2024 = (X_trainval['origin_ts'] >= val_df.index.min())
X_fit, y_fit = X_trainval.loc[~is_2024], y_trainval.loc[~is_2024]
X_cal, y_cal = X_trainval.loc[is_2024], y_trainval.loc[is_2024]

cb = CatBoostQuantileForecaster(iterations=500, early_stopping_rounds=30)
cb.fit(X_fit, y_fit, X_val=X_cal, y_val=y_cal)
cqr_cb = ConformalQuantileRegressor(base=cb, alpha=0.2).calibrate(X_cal, y_cal)

# Per evaluation report (reports/module_b_catboost_calibration_diagnosis.md):
# LightGBM has materially better base calibration than CatBoost, so include
# it here so the production-model choice is defensible on the locked test set.
lgb_m = LightGBMQuantileForecaster(num_boost_round=2000, early_stopping_rounds=30)
lgb_m.fit(X_fit, y_fit, X_val=X_cal, y_val=y_cal)
cqr_lgb = ConformalQuantileRegressor(base=lgb_m, alpha=0.2).calibrate(X_cal, y_cal)

print('Fitted: naive, sn168, catboost+CQR, lightgbm+CQR')
print(f'  CatBoost CQR delta : {cqr_cb.delta:.3f}')
print(f'  LightGBM CQR delta : {cqr_lgb.delta:.3f}')

## Predict on test

In [ ]:
preds = {
    'naive':         naive.predict_quantiles(X_test_full),
    'sn168':         sn168.predict_quantiles(X_test_full),
    'catboost+cqr':  cqr_cb.predict_quantiles(X_test_full),
    'lightgbm+cqr':  cqr_lgb.predict_quantiles(X_test_full),
}
print('Shapes:', {k: v.shape for k, v in preds.items()})

## Headline leaderboard (per horizon group)

In [ ]:
from module_b.features import HORIZON_RANGES, HorizonGroup
from module_b.evaluation import mae, multi_pinball_loss, coverage

rows = []
yt = y_test_full.to_numpy()
for name, q in preds.items():
    for group in HorizonGroup:
        mask = X_test_full[HORIZON_COL].isin(HORIZON_RANGES[group]).to_numpy()
        rows.append({
            'model': name,
            'horizon_group': group.value,
            'mae':         mae(yt[mask], q['q50'].to_numpy()[mask]),
            'pinball_avg': multi_pinball_loss(yt[mask], q.iloc[mask], [0.1, 0.5, 0.9]),
            'coverage_80': coverage(yt[mask], q['q10'].to_numpy()[mask], q['q90'].to_numpy()[mask]),
        })
lb = pd.DataFrame(rows)
lb.set_index(['horizon_group', 'model']).round(3)

## Diebold-Mariano vs seasonal-naive-168h

In [ ]:
from module_b.evaluation import diebold_mariano

yt = y_test_full.to_numpy()
e_sn  = yt - preds['sn168']['q50'].to_numpy()
e_naive = yt - preds['naive']['q50'].to_numpy()
e_cb  = yt - preds['catboost+cqr']['q50'].to_numpy()
e_lgb = yt - preds['lightgbm+cqr']['q50'].to_numpy()

dm_rows = []
for label, e_a, e_b in [
    ('catboost+cqr vs sn168',  e_cb,  e_sn),
    ('lightgbm+cqr vs sn168',  e_lgb, e_sn),
    ('catboost+cqr vs naive',  e_cb,  e_naive),
    ('lightgbm+cqr vs catboost+cqr', e_lgb, e_cb),
]:
    dm = diebold_mariano(e_a, e_b, horizon=24, one_sided=True)
    sig = '***' if dm.p_value < 0.001 else ('**' if dm.p_value < 0.01 else
          ('*' if dm.p_value < 0.05 else ''))
    dm_rows.append({
        'comparison': label,
        'dm_stat':    round(dm.statistic, 3),
        'p_value':    round(dm.p_value, 4),
        'loss_diff':  round(dm.loss_diff_mean, 3),
        'sig':        sig,
    })
dm_df = pd.DataFrame(dm_rows)
print(dm_df.to_string(index=False))
print()
print('Sig codes: *** p<0.001  ** p<0.01  * p<0.05')
print('loss_diff < 0 means the FIRST model has lower squared error than the SECOND.')

## Segment breakdown (CatBoost+CQR)

In [ ]:
from module_b.evaluation import segment_metrics, mae

target_idx = pd.DatetimeIndex(X_test_full['target_ts'])
yp_cb = preds['catboost+cqr']['q50']
seg = segment_metrics(
    target_idx, y_test_full.reset_index(drop=True), yp_cb.reset_index(drop=True), mae,
)
pd.Series(seg).rename('MAE').to_frame().round(3)

## Sample-week visualisation (2025 Q1 test set)

One week of horizon-1 forecasts (2025-02-01 → 2025-02-08) plotted as median + 80% prediction interval against the actual price, for the three strongest models on the leaderboard. The CatBoost+CQR panel is the production model after the per-quantile rewrite.

In [ ]:
import matplotlib.dates as mdates

# Pick one week of horizon-1 forecasts from the 2025 Q1 test window.
test_h1 = (X_test_full[HORIZON_COL] == 1)
win_mask = (
    (X_test_full['target_ts'] >= pd.Timestamp('2025-02-01', tz='UTC'))
    & (X_test_full['target_ts'] <  pd.Timestamp('2025-02-08', tz='UTC'))
    & test_h1
)
ts = X_test_full.loc[win_mask, 'target_ts'].to_numpy()
yt = y_test_full.loc[win_mask].to_numpy()

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, name in zip(axes, ['catboost+cqr', 'lightgbm+cqr', 'sn168']):
    q = preds[name].loc[win_mask].reset_index(drop=True)
    ax.fill_between(ts, q['q10'], q['q90'], alpha=0.3, color='C0', label='80% PI')
    ax.plot(ts, q['q50'], color='C0', lw=1.4, label='median')
    ax.plot(ts, yt, color='k', alpha=0.7, lw=1.0, label='actual')
    cov = ((yt >= q['q10']) & (yt <= q['q90'])).mean()
    mae = float(np.abs(yt - q['q50']).mean())
    ax.set_title(f'{name}    h=1, 2025-02-01 → 2025-02-08    '
                 f'MAE={mae:.1f} €/MWh, coverage={cov:.2f}')
    ax.set_ylabel('€/MWh')
    ax.legend(loc='upper right', fontsize=9)
axes[-1].xaxis.set_major_locator(mdates.DayLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%a %d'))
plt.tight_layout()
plt.show()

## Save final leaderboard

In [ ]:
outputs_dir = REPO_ROOT / 'notebooks' / 'module_b' / 'outputs'
outputs_dir.mkdir(exist_ok=True)
lb.to_parquet(outputs_dir / 'B6_final_leaderboard.parquet')
with (outputs_dir / 'B6_final_leaderboard.md').open('w') as fh:
    fh.write('# Module B - final test-set leaderboard (2025 Q1)\n\n')
    for group in HorizonGroup:
        fh.write(f'## {group.value}\n\n')
        fh.write(lb[lb['horizon_group']==group.value].drop(columns='horizon_group')
                  .set_index('model').round(3).to_markdown())
        fh.write('\n\n')
print('Wrote', outputs_dir / 'B6_final_leaderboard.parquet')
print('Wrote', outputs_dir / 'B6_final_leaderboard.md')